### Query Translation - Decomposition
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some
way as to improve retrival.

### What is Query Decomposition?
**Query decomposition** is a technique of breaking one complex user query into **several** _simpler_, _focused_ sub-queries; retrieving for each, and then combining the results. It’s usually done with an LLM:

1. **Input:** a long or multi-part user question.
2. **Decompose:** use an LLM prompt such as
    “Decompose this question into a list of simpler search queries.”
3. **Retrieve:** run each sub-query against your retriever/vector DB.
4. **Synthesize:** feed the retrieved chunks back into the LLM to build the final answer.

#### Example

**User asks:**

    “Summarize Lilian Weng’s post on LLM agents. Focus on task decomposition methods and explain how Tree of Thoughts extends Chain of Thought.”

A **single vector search** might fail because:
* The query contains **multiple sub-topics** (“task decomposition methods” + “Tree of Thoughts vs CoT”).
* The embedding may dilute meaning across the whole sentence.
**Decomposition:**
* “What are task decomposition methods in Lilian Weng’s LLM agents post?”
* “How does Tree of Thoughts extend Chain of Thought reasoning?”

Retrieve separately, then combine into a coherent answer.

### Why / When to Use Query Decomposition

✅ Use it when:
* **Complex / multi-aspect questions:** 
    e.g., “Compare AutoGPT and BabyAGI, and explain how planning differs from memory.”

* **Broad tasks spanning sub-topics:**
    e.g., “Give me the pros/cons of hybrid search and explain when to use reciprocal rank fusion.”

* **Long, natural language queries:** with multiple clauses joined by “and”, “or”, “how … and also …”.

* **Poor retrieval recall:** when a single embedding search often misses pieces of the question.

🚫 Less helpful when:
* The query is **short and atomic** (e.g., “What is RAG Fusion?”).
* The corpus is tiny or each document already covers the entire topic.

In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [3]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

In [4]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [5]:
retriever = create_or_load_embeddings()

Loading existing embeddings from 
c:\Dev\Code\git-projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_qd

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related 
to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that 
can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Generate just the list of queries. Don't generate any other text, such as numbering or quoting of queries \n
Output ({num_queries} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [12]:
generate_queries = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke(
    {
        "num_queries": 5,
        "question": "What is task decomposition for LLM agents?",
    }
)

['1. "What is task decomposition in AI?"',
 '2. "How do LLM agents use task decomposition?"',
 '3. "Benefits of task decomposition for large language model agents"',
 '4. "Techniques for task decomposition in LLM-based agents"',
 '5. "Examples of task decomposition in LLM agent frameworks"']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [8]:
# from operator import itemgetter

# # RAG
# template = """Answer the following question based on this context:

# {context}

# Question: {question}
# """

# prompt = ChatPromptTemplate.from_template(template)

# # llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# final_rag_chain = (
#     {"context": retrieval_chain, "question": itemgetter("question")}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

# response = final_rag_chain.invoke({"question": question, "num_queries": 5})
# console.print(Markdown(response))